# PDFs

Sign in as the admin test user and read the private Supabase bucket.
Names and sizes only. No Docling.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
os.chdir(ROOT)
for entry in (ROOT / "notebooks", ROOT):
    text = str(entry)
    if text not in sys.path:
        sys.path.insert(0, text)

load_dotenv(ROOT / ".env")

from helpers import classify_uploads, timed

from supabase import create_client

required = [
    "SUPABASE_URL",
    "SUPABASE_ANON_KEY",
    "SUPABASE_ADMIN_EMAIL",
    "SUPABASE_ADMIN_PASSWORD",
]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError("Missing Supabase settings: " + ", ".join(missing))

bucket = os.environ.get("SUPABASE_PDF_BUCKET", "pdfs")
client = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_ANON_KEY"])
client.auth.sign_in_with_password(
    {
        "email": os.environ["SUPABASE_ADMIN_EMAIL"],
        "password": os.environ["SUPABASE_ADMIN_PASSWORD"],
    }
)


def list_objects():
    rows = []
    offset = 0
    while True:
        page = client.storage.from_(bucket).list(
            "",
            {"limit": 100, "offset": offset},
        )
        for item in page:
            metadata = item.get("metadata") or None
            if not metadata:
                continue
            size = metadata.get("size", metadata.get("contentLength", 0)) or 0
            rows.append((item.get("name") or "", int(size)))
        if len(page) < 100:
            return rows
        offset += 100


with timed("list") as listing:
    accepted, rejected = classify_uploads(list_objects())

print(bucket)

In [ ]:
for name, size in accepted:
    print(f"{name}\t{size / (1024 * 1024):.2f} MB\taccepted")

for name, size, reason in rejected:
    print(f"{name}\t{size / (1024 * 1024):.2f} MB\trejected: {reason}")

In [ ]:
print(f"accepted: {len(accepted)}")
print(f"rejected: {len(rejected)}")
print(f"list: {listing['seconds']:.3f}s")

In [ ]:
with timed("read") as reading:
    for name, _size in accepted:
        payload = client.storage.from_(bucket).download(name)
        print(f"{name}\t{len(payload)} bytes")
        del payload

print(f"read: {reading['seconds']:.3f}s")